# Notebook E: Q&A Instruction Fine-Tuning

**Goal**: Fine-tune a QLoRA adapter on top of the Phase 1 merged base model using the Unsloth framework and TRL `SFTTrainer` on Kaggle (T4 GPU).

## 0. Install

In [ ]:

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl" peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done


## 1. Environment Detection & Device Setup

In [ ]:
import os
import sys

# Force single GPU visibility to prevent DDP multi-GPU configuration crashes on Kaggle T4 x2
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Set CUDA_VISIBLE_DEVICES to '0'")

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

if IS_KAGGLE:
    MODEL_NAME = "/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf"
    TRAIN_DS_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset-chatml-template/qa_dataset_train"
    VAL_DS_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset-chatml-template/qa_dataset_val"
    OUTPUT_DIR = "/kaggle/working/qa_checkpoints"
    FINAL_ADAPTER_DIR = "/kaggle/working/spurgeon_lora_qa"
elif IS_COLAB:
    MODEL_NAME = "/content/spurgeon_phase1_merged"
    TRAIN_DS_PATH = "/content/qa_dataset_train"
    VAL_DS_PATH = "/content/qa_dataset_val"
    OUTPUT_DIR = "/content/qa_checkpoints"
    FINAL_ADAPTER_DIR = "/content/spurgeon_lora_qa"
else:
    MODEL_NAME = "../models/spurgeon_phase1_merged"
    TRAIN_DS_PATH = "../data/qa_dataset_train"
    VAL_DS_PATH = "../data/qa_dataset_val"
    OUTPUT_DIR = "../models/qa_checkpoints"
    FINAL_ADAPTER_DIR = "../models/spurgeon_lora_qa"

print(f"Base Model: {MODEL_NAME}")
print(f"Train dataset: {TRAIN_DS_PATH}")

## 2. Load Model & Setup LoRA Adapter

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 2048
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Apply the correct ChatML template for Qwen2.5
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Add PEFT adapter matrices
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,       # Set to 0 to enable Unsloth custom CUDA kernel optimizations
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model and LoRA layers successfully prepared.")

## 3. Load Prepared Datasets

In [ ]:
import shutil
from datasets import load_from_disk

# Kaggle read-only mapping fallback: Copy dataset to working directory before mapping tokenization caches
LOCAL_TRAIN_PATH = "/kaggle/working/qa_dataset_train_local" if IS_KAGGLE else TRAIN_DS_PATH
LOCAL_VAL_PATH = "/kaggle/working/qa_dataset_val_local" if IS_KAGGLE else VAL_DS_PATH

if IS_KAGGLE:
    if os.path.exists(LOCAL_TRAIN_PATH):
        shutil.rmtree(LOCAL_TRAIN_PATH)
    shutil.copytree(TRAIN_DS_PATH, LOCAL_TRAIN_PATH)
    if os.path.exists(LOCAL_VAL_PATH):
        shutil.rmtree(LOCAL_VAL_PATH)
    shutil.copytree(VAL_DS_PATH, LOCAL_VAL_PATH)
    print("Copied datasets to local workspace for read-write compatibility.")

train_ds = load_from_disk(LOCAL_TRAIN_PATH)
val_ds = load_from_disk(LOCAL_VAL_PATH)

print(f"Loaded {len(train_ds):,} train examples and {len(val_ds):,} val examples.")

## 4. Training Arguments & SFT Config

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # Effective batch size = 16
    num_train_epochs=3,                 # 3 epochs is sufficient for SFT
    warmup_steps=50,
    learning_rate=1e-4,                 # Fine-tuning rate (lower than Phase 1 pre-train)
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,                      # IMPORTANT: Do not pack Q&A pairs together to keep distinct boundary contexts
    seed=42,
)

## 5. Initialize Trainer & Apply Serialization Fix

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

# Apply SFTConfig serialization/pickling crash fix (modules mismatch workaround)
try:
    import sys
    if 'trl.trainer.sft_config' in sys.modules:
        sys.modules['trl.trainer.sft_config'].SFTConfig = trainer.args.__class__
        print("Successfully bound SFTConfig serialization hook.")
except Exception as e:
    print(f"Warning binding serialization hook: {e}")

## 6. Execute Training

In [ ]:
# Execute training run
trainer_stats = trainer.train()

print(f"Training finished. Time: {trainer_stats.metrics['train_runtime']:.2f}s")

## 7. Save PEFT Adapter

In [ ]:
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Successfully saved LoRA adapter weights to {FINAL_ADAPTER_DIR}")